<a href="https://colab.research.google.com/github/Thashmikab/london-smart-meter-analysis/blob/main/notebooks/02_preparing_for_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Preparing the data for analysis

Notebook 01 produced a clean table of 167.5 million half-hourly meter readings. On its own it can't answer the questions this project is after: did the time-of-use tariff change behaviour, did it save money, and did it cut carbon? Answering those needs four more sources, a decision about which households have enough data for each analysis, and a daily table that brings it all together.

| Source | What it adds | Where it comes from |
|---|---|---|
| Household information | Tariff group (standard or time-of-use) and ACORN socio-economic group | Kaggle |
| Hourly weather | Temperature, humidity, wind speed | Kaggle (Dark Sky) |
| Bank holidays | Days off that behave like weekends | Rebuilt from the official England list (the Kaggle file was wrong) |
| 2013 tariff schedule | High, Normal or Low price for every half-hour of 2013 | `Tariffs.xlsx`, London Datastore |
| GB generation mix | Grid carbon intensity and output by fuel, every half-hour | NESO data portal API |

**What this notebook found and decided**

- The bank holiday file had **day and month swapped** for every date where both are 12 or less (Good Friday 2012 sat in June), and it had no 2011 holidays. It was replaced with the official list, and the repair was cross-checked against it.
- The weather is in **UTC/GMT** like the meter data, with **two missing hours** (9–10 September 2013), filled and flagged. Columns no analysis uses, including pressure with its 13 nulls, were dropped.
- The tariff schedule and generation mix are **complete**: no gaps, duplicates or nulls.
- Households joined the trial in **two waves**, mostly April–July 2012 and again in September–October 2012, in the same proportions for both tariff groups. That shaped the comparison design: **July–December 2012 against July–December 2013**.
- Households are **flagged per analysis, not deleted**: 1,062 time-of-use and 4,146 standard households qualify for the tariff comparison; 1,088 and 4,230 for the 2013 bill and carbon analysis.
- Meter timestamps mark the **end** of each half-hour, settled from the data by testing both alignments against the price events.
- A first look already shows time-of-use households **using less during high-price half-hours and more during low-price ones** than standard households.

**Outputs** (saved to Google Drive for the later notebooks): `households`, `household_day` (3,490,380 rows with daily kWh, evening-peak kWh, CO₂ and costs), `weather_hourly`, `bank_holidays`, `tariffs_2013`, `generation_mix`.

**To run:** notebook 01 must have saved `halfhourly_cleaned` to `MyDrive/london-smart-meters/`. Download `Tariffs.xlsx` from the [London Datastore](https://data.london.gov.uk/dataset/smartmeter-energy-consumption-data-in-london-households-vqm0d) into `MyDrive/`. Kaggle credentials go in Colab Secrets as `KAGGLE_USER` and `KAGGLE_API_TOKEN`.

## Setup

In [1]:
!pip install pyspark

In [10]:
import os
from google.colab import userdata
import pandas as pd
import pyspark
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, Window
from pyspark.sql import types as T
from pyspark.sql.functions import col, count, when
from pyspark.sql.types import DoubleType
from google.colab import drive

In [2]:
#@title Start Spark

spark = SparkSession.builder.appName("london-smart-meters-prep").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

## 1. The cleaned readings from notebook 01

Colab wipes its local disk when a session ends, so notebook 01 saved its output to Google Drive. Copying it back to local disk before reading is faster than reading through the Drive mount. The check: 167,538,240 rows and six columns, including the three flags.

In [4]:
#@title Get the cleaned readings from Google Drive

drive.mount("/content/drive")
!mkdir -p data/parquet
!cp -r /content/drive/MyDrive/london-smart-meters/halfhourly_cleaned data/parquet/

hh = spark.read.parquet("data/parquet/halfhourly_cleaned")
print(f"{hh.count():,} rows")      # should be 167,538,240
hh.printSchema()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
167,538,240 rows
root
 |-- household_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- imputed: boolean (nullable = true)
 |-- suspect_spike: boolean (nullable = true)
 |-- zero_day: boolean (nullable = true)



## 2. The other sources

**Kaggle files.** The household table, hourly weather and bank holidays come with the Kaggle dataset.

**The 2013 tariff schedule.** The time-of-use households were sent day-ahead price signals in three bands: High (67.20p/kWh), Normal (11.76p/kWh) and Low (3.99p/kWh). Standard households paid a flat 14.228p/kWh. `Tariffs.xlsx` lists the band for every half-hour of 2013: 17,520 rows, of which 788 are High and 1,660 Low.

**The GB generation mix.** NESO publishes the historic half-hourly generation mix, with a carbon intensity column, through an open API that needs no key. The trial period is fetched in four half-year chunks so no single request is capped, and saved to Drive so the API is only called once. All four chunks match their day counts exactly: 1 November 2011 to 1 March 2014 is 851 days × 48 = 40,848 half-hours.

In [6]:
#@title  Download the dataset

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USER')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')





# Create the standard project folders
!mkdir -p data src notebooks
print("Project folders 'data/', 'src/', and 'notebooks/' are ready.")




# Download the dataset directly into the data folder and unzip it
!kaggle datasets download -d jeanmidev/smart-meters-in-london -p data --unzip
print("Dataset downloaded and extracted into the 'data/' folder!")




!ls -l data

Project folders 'data/', 'src/', and 'notebooks/' are ready.
Dataset URL: https://www.kaggle.com/datasets/jeanmidev/smart-meters-in-london
License(s): ODbL-1.0
100% 1.17G/1.17G [00:13<00:00, 92.0MB/s]

Dataset downloaded and extracted into the 'data/' folder!
total 346324
-rw-r--r-- 1 root root    130703 Oct  5 13:49 acorn_details.csv
drwxr-xr-x 3 root root      4096 Oct  5 13:49 daily_dataset
-rw-r--r-- 1 root root 351799634 Oct  5 13:49 daily_dataset.csv
-rw-r--r-- 1 root root     63096 Oct  5 13:49 darksky_parameters_documentation.html
drwxr-xr-x 3 root root      4096 Oct  5 13:49 halfhourly_dataset
drwxr-xr-x 3 root root      4096 Oct  5 13:50 hhblock_dataset
-rw-r--r-- 1 root root    234595 Oct  5 13:50 informations_households.csv
drwxr-xr-x 3 root root      4096 Oct  5 12:38 parquet
-rw-r--r-- 1 root root       786 Oct  5 13:50 uk_bank_holidays.csv
-rw-r--r-- 1 root root    341198 Oct  5 13:50 weather_daily_darksky.csv
-rw-r--r-- 1 root root   2034495 Oct  5 13:50 weather_hourly_

In [8]:
tariffs_pd = pd.read_excel("/content/drive/MyDrive/Tariffs.xlsx")
print(tariffs_pd.shape)
print(tariffs_pd.dtypes)
print(tariffs_pd.head(10))
print(tariffs_pd.iloc[:, 1].value_counts())   # how many half-hours at each price band

tariffs = spark.createDataFrame(tariffs_pd)

(17520, 2)
TariffDateTime    datetime64[ns]
Tariff                    object
dtype: object
       TariffDateTime  Tariff
0 2013-01-01 00:00:00  Normal
1 2013-01-01 00:30:00  Normal
2 2013-01-01 01:00:00  Normal
3 2013-01-01 01:30:00  Normal
4 2013-01-01 02:00:00  Normal
5 2013-01-01 02:30:00  Normal
6 2013-01-01 03:00:00  Normal
7 2013-01-01 03:30:00  Normal
8 2013-01-01 04:00:00  Normal
9 2013-01-01 04:30:00  Normal
Tariff
Normal    15072
Low        1660
High        788
Name: count, dtype: int64


In [11]:
RES = "f93d1835-75bc-43e5-84ad-12472b180a98"      # Historic GB Generation Mix resource
URL = "https://api.neso.energy/api/3/action/datastore_search_sql"

periods = [("2011-11-01", "2012-07-01"), ("2012-07-01", "2013-01-01"),
           ("2013-01-01", "2013-07-01"), ("2013-07-01", "2014-03-01")]

frames = []
for start, end in periods:
    sql = f'SELECT * FROM "{RES}" WHERE "DATETIME" >= \'{start}\' AND "DATETIME" < \'{end}\''
    r = requests.get(URL, params={"sql": sql}, timeout=120)
    r.raise_for_status()
    rows = r.json()["result"]["records"]
    print(start, "->", end, len(rows), "rows")
    frames.append(pd.DataFrame(rows))

genmix = pd.concat(frames, ignore_index=True)
print(genmix.shape)
print(list(genmix.columns))

genmix.to_csv("/content/drive/MyDrive/london-smart-meters/neso_generation_mix_2011_2014.csv", index=False)

2011-11-01 -> 2012-07-01 11664 rows
2012-07-01 -> 2013-01-01 8832 rows
2013-01-01 -> 2013-07-01 8688 rows
2013-07-01 -> 2014-03-01 11664 rows
(40848, 36)
['RENEWABLE_perc', 'NUCLEAR', 'HYDRO_perc', 'STORAGE', 'IMPORTS_perc', 'SOLAR_perc', 'LOW_CARBON', 'BIOMASS_perc', 'HYDRO', 'STORAGE_perc', 'WIND_EMB', 'WIND_perc', 'IMPORTS', 'LOW_CARBON_perc', 'GAS', 'COAL_perc', 'CARBON_INTENSITY', 'COAL', 'FOSSIL', 'ZERO_CARBON_perc', 'FOSSIL_perc', 'GENERATION_perc', 'GENERATION', 'DATETIME', 'OTHER', 'NUCLEAR_perc', 'ZERO_CARBON', 'WIND_EMB_perc', '_full_text', 'BIOMASS', 'GAS_perc', 'OTHER_perc', 'WIND', 'SOLAR', '_id', 'RENEWABLE']


## 3. Load everything and check the schemas

The tables are small, so `inferSchema` is fine here. The check that matters is how each **time column** was read, because every join later is on time. All of them came through correctly: weather `time`, `TariffDateTime` and NESO `DATETIME` as timestamps, bank holidays as dates. The session timezone is set to UTC so pandas datetimes aren't shifted on their way into Spark; the tariff schedule starting at exactly 00:00 on 1 January 2013 confirms it worked.

A correct type isn't the same as a correct value, though. The bank holiday dates parsed without complaint and were still wrong (section 5).

In [12]:
#@title Load the details and check the schema


# Initialize your session and load with header configuration
spark = SparkSession.builder.appName('Dataframe').getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")   # so pandas datetimes aren't shifted on the way into Spark


# Kaggle files
houseHold_df = spark.read.csv('/content/data/informations_households.csv', header=True, inferSchema=True)
WeatherHourly_df = spark.read.csv('/content/data/weather_hourly_darksky.csv', header=True, inferSchema=True)
bankHoliday_df = spark.read.csv('/content/data/uk_bank_holidays.csv', header=True, inferSchema=True)




# Tariff schedule (Excel, from the London Datastore)
tariff_df = spark.createDataFrame(pd.read_excel("/content/drive/MyDrive/Tariffs.xlsx"))






# NESO generation mix (saved from the API), dropping the two API bookkeeping columns
genMix_df = (spark.read.csv("/content/drive/MyDrive/london-smart-meters/neso_generation_mix_2011_2014.csv", header=True, inferSchema=True).drop("_id", "_full_text"))








# Display the top row to verify
display(houseHold_df.head())
houseHold_df.show(3)

display(WeatherHourly_df.head())
WeatherHourly_df.show(3)

display(bankHoliday_df.head())
bankHoliday_df.show(3)

display(tariff_df.head())
tariff_df.show(3)

display(genMix_df.head())
genMix_df.show(3)


houseHold_df.printSchema()
WeatherHourly_df.printSchema()
bankHoliday_df.printSchema()
tariff_df.printSchema()
genMix_df.printSchema()

Row(LCLid='MAC005492', stdorToU='ToU', Acorn='ACORN-', Acorn_grouped='ACORN-', file='block_0')

+---------+--------+-------+-------------+-------+
|    LCLid|stdorToU|  Acorn|Acorn_grouped|   file|
+---------+--------+-------+-------------+-------+
|MAC005492|     ToU| ACORN-|       ACORN-|block_0|
|MAC001074|     ToU| ACORN-|       ACORN-|block_0|
|MAC000002|     Std|ACORN-A|     Affluent|block_0|
+---------+--------+-------+-------------+-------+
only showing top 3 rows


Row(visibility=5.97, windBearing=104, temperature=10.24, time=datetime.datetime(2011, 11, 11, 0, 0), dewPoint=8.86, pressure=1016.76, apparentTemperature=10.24, windSpeed=2.77, precipType='rain', icon='partly-cloudy-night', humidity=0.91, summary='Partly Cloudy')

+----------+-----------+-----------+-------------------+--------+--------+-------------------+---------+----------+-------------------+--------+-------------+
|visibility|windBearing|temperature|               time|dewPoint|pressure|apparentTemperature|windSpeed|precipType|               icon|humidity|      summary|
+----------+-----------+-----------+-------------------+--------+--------+-------------------+---------+----------+-------------------+--------+-------------+
|      5.97|        104|      10.24|2011-11-11 00:00:00|    8.86| 1016.76|              10.24|     2.77|      rain|partly-cloudy-night|    0.91|Partly Cloudy|
|      4.88|         99|       9.76|2011-11-11 01:00:00|    8.83| 1016.63|               8.24|     2.95|      rain|partly-cloudy-night|    0.94|Partly Cloudy|
|       3.7|         98|       9.46|2011-11-11 02:00:00|    8.79| 1016.36|               7.76|     3.17|      rain|partly-cloudy-night|    0.96|Partly Cloudy|
+----------+-----------+-----------+----------

Row(Bank holidays=datetime.date(2012, 12, 26), Type='Boxing Day')

+-------------+-------------------+
|Bank holidays|               Type|
+-------------+-------------------+
|   2012-12-26|         Boxing Day|
|   2012-12-25|      Christmas Day|
|   2012-08-27|Summer bank holiday|
+-------------+-------------------+
only showing top 3 rows


Row(TariffDateTime=datetime.datetime(2013, 1, 1, 0, 0), Tariff='Normal')

+-------------------+------+
|     TariffDateTime|Tariff|
+-------------------+------+
|2013-01-01 00:00:00|Normal|
|2013-01-01 00:30:00|Normal|
|2013-01-01 01:00:00|Normal|
+-------------------+------+
only showing top 3 rows


Row(RENEWABLE_perc=7.0, NUCLEAR=6826.0, HYDRO_perc=2.6, STORAGE=0, IMPORTS_perc=5.0, SOLAR_perc=0.0, LOW_CARBON=8921.0, BIOMASS_perc=0.0, HYDRO=787, STORAGE_perc=0.0, WIND_EMB=306.0, WIND_perc=3.3, IMPORTS=1494.0, LOW_CARBON_perc=29.7, GAS=10366.0, COAL_perc=30.7, CARBON_INTENSITY=434.0, COAL=9209.0, FOSSIL=19575.0, ZERO_CARBON_perc=30.6, FOSSIL_perc=65.3, GENERATION_perc=100.0, GENERATION=29990.0, DATETIME=datetime.datetime(2011, 11, 1, 0, 0), OTHER=0.0, NUCLEAR_perc=22.8, ZERO_CARBON=8615.0, WIND_EMB_perc=1.0, BIOMASS=0.0, GAS_perc=34.6, OTHER_perc=0.0, WIND=1002.0, SOLAR=0.0, RENEWABLE=2095.0)

+--------------+-------+----------+-------+------------+----------+----------+------------+-----+------------+--------+---------+-------+---------------+-------+---------+----------------+------+-------+----------------+-----------+---------------+----------+-------------------+-----+------------+-----------+-------------+-------+--------+----------+------+-----+---------+
|RENEWABLE_perc|NUCLEAR|HYDRO_perc|STORAGE|IMPORTS_perc|SOLAR_perc|LOW_CARBON|BIOMASS_perc|HYDRO|STORAGE_perc|WIND_EMB|WIND_perc|IMPORTS|LOW_CARBON_perc|    GAS|COAL_perc|CARBON_INTENSITY|  COAL| FOSSIL|ZERO_CARBON_perc|FOSSIL_perc|GENERATION_perc|GENERATION|           DATETIME|OTHER|NUCLEAR_perc|ZERO_CARBON|WIND_EMB_perc|BIOMASS|GAS_perc|OTHER_perc|  WIND|SOLAR|RENEWABLE|
+--------------+-------+----------+-------+------------+----------+----------+------------+-----+------------+--------+---------+-------+---------------+-------+---------+----------------+------+-------+----------------+-----------+---------------+

Two tidy-ups: the bank holiday column name contains a space, and the generation mix has 34 columns, of which the analysis needs the timestamp, carbon intensity, total generation and output per fuel in MW. The percentage columns can be recalculated from those if ever needed.

In [13]:
#@title Tidy up names and columns

bankHoliday_df = bankHoliday_df.withColumnRenamed("Bank holidays", "date").withColumnRenamed("Type", "holiday")

genMix_df = genMix_df.select("DATETIME", "CARBON_INTENSITY", "GENERATION",
                             "GAS", "COAL", "NUCLEAR", "WIND", "WIND_EMB", "SOLAR",
                             "HYDRO", "BIOMASS", "IMPORTS", "STORAGE", "OTHER")

## 4. Quality checks: nulls, duplicates, time range, gaps

The same ideas as notebook 01, applied to small tables. For each time-indexed table: null counts, duplicate timestamps, first and last timestamp, and whether every expected step between them is present (hourly for weather, half-hourly for the others).

| Table | Rows | Duplicates | Missing steps | Nulls |
|---|---:|---:|---:|---|
| Weather (hourly) | 21,165 | 0 | **2** | 13 in `pressure` |
| Tariffs 2013 (half-hourly) | 17,520 | 0 | 0 | 0 |
| Generation mix (half-hourly) | 40,848 | 0 | 0 | 0 |

The weather covers 1 November 2011 to 31 March 2014, so the whole meter period.

In [14]:
#@title Check each table: time range, duplicates, gaps, nulls

def check_time_table(df, time_col, step_minutes, name):

  # Check the nulls

  df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()



  # Check the dupes

  dupes = df.groupBy(time_col).count().filter("count > 1").count()
  print(f"Duplicate timestamps: {dupes:,}")




  # Check plausibility of the time range

  df.agg(F.count("*").alias("rows"),
         F.min(time_col).alias("first"),
         F.max(time_col).alias("last")).show(truncate=False)




  # Gaps: compare with the full expected grid between first and last timestamp
  first, last = df.agg(F.min(time_col), F.max(time_col)).first()
  expected = int((last - first).total_seconds() // (step_minutes * 60)) + 1
  actual = df.select(time_col).distinct().count()
  print(f"Expected {expected:,} timestamps, found {actual:,}, missing {expected - actual:,}")

  # Off-grid timestamps
  off = df.filter((F.minute(time_col) % step_minutes != 0) | (F.second(time_col) != 0)).count()
  print(f"Off-grid timestamps: {off:,}")






check_time_table(WeatherHourly_df, "time", 60, "Weather (hourly)")
check_time_table(tariff_df, "TariffDateTime", 30, "Tariffs (half-hourly, 2013)")
check_time_table(genMix_df, "DATETIME", 30, "Generation mix (half-hourly)")

print("\n===== Bank holidays =====")
bankHoliday_df.orderBy("date").show(50, truncate=False)

+----------+-----------+-----------+----+--------+--------+-------------------+---------+----------+----+--------+-------+
|visibility|windBearing|temperature|time|dewPoint|pressure|apparentTemperature|windSpeed|precipType|icon|humidity|summary|
+----------+-----------+-----------+----+--------+--------+-------------------+---------+----------+----+--------+-------+
|         0|          0|          0|   0|       0|      13|                  0|        0|         0|   0|       0|      0|
+----------+-----------+-----------+----+--------+--------+-------------------+---------+----------+----+--------+-------+

Duplicate timestamps: 0
+-----+-------------------+-------------------+
|rows |first              |last               |
+-----+-------------------+-------------------+
|21165|2011-11-01 00:00:00|2014-03-31 22:00:00|
+-----+-------------------+-------------------+

Expected 21,167 timestamps, found 21,165, missing 2
Off-grid timestamps: 0
+--------------+------+
|TariffDateTime|Tari

## 5. Bank holidays: the file had the dates wrong

The list printed above looks plausible until the dates are read against their labels: Good Friday 2012 on 4 June, Easter Monday on 4 September, New Year's Day (substitute) on 1 February. **Day and month are swapped for every date where both are 12 or less.** Dates such as 25 December or 29 March are right, because a day above 12 can't be a month. Reading the file as plain text shows the swap is in the file itself, not caused by Spark's parsing, and the file also has **no 2011 holidays**, although the meter data starts in November 2011.

So the bank holidays are rebuilt from the official England list (the `holidays` package), which covers 2011–2014. The Kaggle file is repaired with a simple rule (swap day and month when both are 12 or less) and cross-checked: every repaired date appears in the official list, so the rule is confirmed. The official list is what the rest of the project uses.

In [ ]:
#@title Look at the raw bank holiday dates

bankHoliday_raw = spark.read.csv('/content/data/uk_bank_holidays.csv', header=True,
                                 inferSchema=False, encoding="iso-8859-1")
bankHoliday_raw.show(30, truncate=False)

In [19]:
#@title Bank holidays: rebuild from an authoritative source, cross-check the file

!pip install -q holidays
import holidays

eng = holidays.country_holidays("GB", subdiv="ENG", years=range(2011, 2015))
holiday_df = spark.createDataFrame(pd.DataFrame(sorted(eng.items()), columns=["date", "holiday"]))
holiday_df.show(40, truncate=False)

# Repair the Kaggle file: swap day and month when both are 12 or less
parts = F.split(col("Bank holidays"), "-")
file_fixed = bankHoliday_raw.withColumn("date",
    F.when((parts[1].cast("int") <= 12) & (parts[2].cast("int") <= 12),
           F.to_date(F.concat_ws("-", parts[0], parts[2], parts[1])))
     .otherwise(F.to_date(col("Bank holidays"))))

# Every repaired date should appear in the official list
print("Repaired file dates not in the official list:")
file_fixed.join(holiday_df, "date", "left_anti").show(truncate=False)

+----------+--------------------------------+
|date      |holiday                         |
+----------+--------------------------------+
|2011-01-01|New Year's Day                  |
|2011-01-03|New Year's Day (observed)       |
|2011-04-22|Good Friday                     |
|2011-04-25|Easter Monday                   |
|2011-04-29|Wedding of William and Catherine|
|2011-05-02|May Day                         |
|2011-05-30|Spring Bank Holiday             |
|2011-08-29|Late Summer Bank Holiday        |
|2011-12-25|Christmas Day                   |
|2011-12-26|Boxing Day                      |
|2011-12-27|Christmas Day (observed)        |
|2012-01-01|New Year's Day                  |
|2012-01-02|New Year's Day (observed)       |
|2012-04-06|Good Friday                     |
|2012-04-09|Easter Monday                   |
|2012-05-07|May Day                         |
|2012-06-04|Spring Bank Holiday             |
|2012-06-05|Diamond Jubilee of Elizabeth II |
|2012-08-27|Late Summer Bank Holid

## 6. Weather: two missing hours, and only the columns the analysis needs

The two missing hours are 23:00 on 9 September and 00:00 on 10 September 2013. Neither is a clock-change hour; none of the three spring clock changes in the period is missing, and the autumn ones produced no duplicates. That points to the weather being in UTC/GMT, the same as the meter data, so the two line up.

Weather earns its place through **temperature**, the main driver of household demand and something the 2012-against-2013 comparison has to account for. Humidity and wind speed are kept as cheap extras. Pressure, visibility, wind bearing, dew point and the text summaries don't serve any question here, so they're dropped, and the 13 null pressures go with them.

The missing hours are **rows that don't exist**, not nulls, so a null-filling step alone would change nothing. The weather is first joined onto the full hourly grid, which creates the two rows with nulls; each gap is then filled with the average of the readings either side and flagged as `interpolated`. Rows go from 21,165 to 21,167, with no nulls left.

In [18]:
#@title Which weather hours are missing, and where are the null pressures?

hours = spark.sql("""
    SELECT explode(sequence(timestamp'2011-11-01 00:00:00',
                            timestamp'2014-03-31 22:00:00',
                            interval 1 hour)) AS time""")

hours.join(WeatherHourly_df, "time", "left_anti").show()

WeatherHourly_df.filter(col("pressure").isNull()).select("time").show(13)

+-------------------+
|               time|
+-------------------+
|2013-09-09 23:00:00|
|2013-09-10 00:00:00|
+-------------------+

+-------------------+
|               time|
+-------------------+
|2012-05-07 18:00:00|
|2012-10-26 15:00:00|
|2013-09-13 11:00:00|
|2013-01-03 07:00:00|
|2013-08-25 23:00:00|
|2013-08-26 00:00:00|
|2013-08-16 01:00:00|
|2013-08-01 00:00:00|
|2013-06-09 13:00:00|
|2013-09-09 01:00:00|
|2013-09-09 04:00:00|
|2014-03-22 20:00:00|
|2014-03-22 21:00:00|
+-------------------+



In [25]:
#@title Weather: keep what the analysis needs, fill the two missing hours

WeatherHourly_df = WeatherHourly_df.select("time", "temperature", "humidity", "windSpeed")

# 1. Put the weather on the full hourly grid, so the 2 missing hours become rows with nulls
df = (hours.join(WeatherHourly_df, "time", "left")
           .withColumn("interpolated", F.col("temperature").isNull()))

# 2. Set up windows looking back and forward from each row
window_prev = Window.orderBy("time").rowsBetween(Window.unboundedPreceding, -1)
window_next = Window.orderBy("time").rowsBetween(1, Window.unboundedFollowing)

# 3. Fill each gap with the average of the last value before it and the first value after it
for col_name in ["temperature", "humidity", "windSpeed"]:
    last_val = F.last(col_name, ignorenulls=True).over(window_prev)
    next_val = F.first(col_name, ignorenulls=True).over(window_next)
    df = df.withColumn(col_name, F.coalesce(F.col(col_name), (last_val + next_val) / 2))

WeatherHourly_df = df

# 4. Check it worked
print(f"Rows: {WeatherHourly_df.count():,}")     # should be 21,167 (was 21,165)
WeatherHourly_df.select([count(when(col(c).isNull(), c)).alias(c) for c in WeatherHourly_df.columns]).show()
WeatherHourly_df.filter("interpolated").show()  # the 2 filled hours

Rows: 21,167
+----+-----------+--------+---------+------------+
|time|temperature|humidity|windSpeed|interpolated|
+----+-----------+--------+---------+------------+
|   0|          0|       0|        0|           0|
+----+-----------+--------+---------+------------+

+-------------------+------------------+--------+------------------+------------+
|               time|       temperature|humidity|         windSpeed|interpolated|
+-------------------+------------------+--------+------------------+------------+
|2013-09-09 23:00:00|10.344999999999999|   0.905|3.0549999999999997|        true|
|2013-09-10 00:00:00|10.344999999999999|   0.905|3.0549999999999997|        true|
+-------------------+------------------+--------+------------------+------------+



## 7. Are the values plausible?

- **Temperature** −5.6 to 32.4 °C (the July 2013 heatwave at the top end), **humidity** 0.23–1.0, **wind speed** never negative.
- **Carbon intensity** 259 to 644 gCO₂/kWh, median 493. The grid was coal-heavy in 2011–2014: coal's median output (15.1 GW) was well above gas (9.2 GW). The dirtiest half-hours emitted about two and a half times as much per kWh as the cleanest, which is why *when* electricity is used matters for carbon.
- **Total generation** 21–59 GW, no negative output for any fuel, and `GENERATION` equals the sum of the fuel columns to within 2 MW (rounding).
- **Tariffs:** exactly the three expected bands.

In [26]:
#@title Are the values plausible?

# Weather: London temperatures roughly -10 to 35 °C, humidity a fraction 0-1, wind speed not negative
WeatherHourly_df.select("temperature", "humidity", "windSpeed").summary("min", "1%", "50%", "99%", "max").show()

# Generation mix: no negative generation, carbon intensity in a sensible range for 2011-2014
genMix_df.select("CARBON_INTENSITY", "GENERATION", "GAS", "COAL", "NUCLEAR", "WIND", "IMPORTS") \
         .summary("min", "1%", "50%", "99%", "max").show()

fuels = ["GAS", "COAL", "NUCLEAR", "WIND", "WIND_EMB", "SOLAR", "HYDRO", "BIOMASS", "IMPORTS", "STORAGE", "OTHER"]
neg = genMix_df.select([F.sum((col(c) < 0).cast("int")).alias(c) for c in fuels])
neg.show()                                                     # negative values per fuel

# Does GENERATION equal the sum of the fuels?
gm = genMix_df.withColumn("fuel_sum", sum(col(c) for c in fuels)) \
              .withColumn("diff", col("GENERATION") - col("fuel_sum"))
gm.select("diff").summary("min", "50%", "max").show()

# Tariffs: only the three expected bands?
tariff_df.groupBy("Tariff").count().show()

+-------+-----------+--------+---------+
|summary|temperature|humidity|windSpeed|
+-------+-----------+--------+---------+
|    min|      -5.64|    0.23|     0.04|
|     1%|      -0.92|     0.4|     0.45|
|    50%|       9.93|    0.81|     3.68|
|    99%|      25.39|    0.98|     9.66|
|    max|       32.4|     1.0|     14.8|
+-------+-----------+--------+---------+

+-------+----------------+----------+-------+-------+-------+------+-------+
|summary|CARBON_INTENSITY|GENERATION|    GAS|   COAL|NUCLEAR|  WIND|IMPORTS|
+-------+----------------+----------+-------+-------+-------+------+-------+
|    min|           259.0|   21348.0| 1556.0| 3223.0| 4099.0|   8.0|    0.0|
|     1%|           344.0|   23605.0| 2112.0| 6636.0| 4793.0|  97.0|    0.0|
|    50%|           493.0|   37400.0| 9167.0|15142.0| 7549.0|1642.0| 1986.0|
|    99%|           597.0|   52913.0|19365.0|23569.0| 9197.0|5592.0| 2996.0|
|    max|           644.0|   59403.0|22985.0|25197.0| 9342.0|6215.0| 3366.0|
+-------+-----

## 8. Households: who has enough data for which analysis

**The household table** has one row per household (5,566, no duplicate IDs): 4,443 standard and 1,123 time-of-use. 49 households have the ACORN code `ACORN-U` (unclassified) and 2 a blank `ACORN-`. Their meter data is valid, so they're kept and recoded as one `Unknown` group, which ACORN comparisons can leave out.

In [29]:
#@title Clean the household table

households_clean = (houseHold_df
    .select(F.col("LCLid").alias("household_id"),
            F.col("stdorToU").alias("tariff"),
            F.when(F.col("Acorn").isin("ACORN-U", "ACORN-"), "Unknown")
             .otherwise(F.col("Acorn")).alias("acorn"),
            F.when(F.col("Acorn_grouped").isin("ACORN-U", "ACORN-"), "Unknown")
             .otherwise(F.col("Acorn_grouped")).alias("acorn_group")))

households_clean.groupBy("tariff").count().show()          # Std 4,443 / ToU 1,123
households_clean.groupBy("acorn_group").count().show()     # Unknown should be 51

+------+-----+
|tariff|count|
+------+-----+
|   Std| 4443|
|   ToU| 1123|
+------+-----+

+-----------+-----+
|acorn_group|count|
+-----------+-----+
|  Adversity| 1816|
|   Affluent| 2192|
|    Unknown|   51|
|Comfortable| 1507|
+-----------+-----+



**Coverage within each household's time in the trial is excellent** (median 0.998, 1st percentile 0.96): once a meter was installed it reported reliably. **What varies is when households were in the trial.** The median household has 229 complete days in 2012 against 365 in 2013, because most meters were installed partway through 2012.

**Zero days belong to a few households.** The 75th percentile is 0. MAC004067 recorded zero on all 646 of its days; others have long dead stretches but real days too. Zero days are excluded day by day rather than removing whole households.

In [27]:
#@title Coverage per household

hh_days = (hh.groupBy("household_id", F.to_date("timestamp").alias("day"))
             .agg(F.max(F.col("zero_day").cast("int")).alias("zero_day")))

coverage = (hh_days.groupBy("household_id")
    .agg(F.count("*").alias("complete_days"),
         F.sum("zero_day").alias("zero_days"),
         F.min("day").alias("first_day"),
         F.max("day").alias("last_day"),
         F.sum((F.year("day") == 2012).cast("int")).alias("days_2012"),
         F.sum((F.year("day") == 2013).cast("int")).alias("days_2013"))
    .withColumn("span_days", F.datediff("last_day", "first_day") + 1)
    .withColumn("coverage", F.round(F.col("complete_days") / F.col("span_days"), 3))
    .withColumn("usable_days", F.col("complete_days") - F.col("zero_days")))

coverage.select("complete_days", "usable_days", "coverage", "zero_days", "days_2012", "days_2013") \
        .summary("min", "1%", "5%", "25%", "50%", "max").show()

coverage.orderBy("usable_days").show(10)       # the thinnest households
coverage.orderBy(F.desc("zero_days")).show(10) # the worst zero-day households

+-------+-------------+-----------+--------+---------+---------+---------+
|summary|complete_days|usable_days|coverage|zero_days|days_2012|days_2013|
+-------+-------------+-----------+--------+---------+---------+---------+
|    min|            7|          0|   0.698|        0|        0|        0|
|     1%|          236|        214|    0.96|        0|       66|       29|
|     5%|          434|        423|   0.992|        0|       76|      220|
|    25%|          597|        595|   0.997|        0|      189|      364|
|    50%|          647|        646|   0.998|        0|      229|      365|
|    max|          827|        827|     1.0|      783|      366|      365|
+-------+-------------+-----------+--------+---------+---------+---------+

+------------+-------------+---------+----------+----------+---------+---------+---------+--------+-----------+
|household_id|complete_days|zero_days| first_day|  last_day|days_2012|days_2013|span_days|coverage|usable_days|
+------------+-----------

**The comparison design.** Both groups joined in the same proportions month by month (roughly one new household in five was time-of-use), so timing doesn't bias the group comparison. Most were installed by July 2012, with a second wave in September–October 2012. The comparison therefore uses **July–December 2012 as the baseline and July–December 2013** with the tariff in effect: the same season in both years, so winter heating can't pass for a tariff effect.

| Minimum usable days in each half-year | Std | ToU |
|---|---:|---:|
| 30 | 4,195 | 1,074 |
| 60 | 4,146 | 1,062 |
| 90 | 3,757 | 959 |
| 120 | 3,514 | 894 |
| 150 | 3,434 | 884 |

The drop between 60 and 120 days is the September–October wave, which joined too late for a long 2012 baseline. **60 days** is the main threshold, keeping 95% of time-of-use households; 120 days is kept available as a robustness check in notebook 03. For the 2013 bill and carbon analysis, which uses per-day averages, a household needs **180 usable days in 2013**.

In [31]:
#@title Usable days per comparison period, by tariff group

usable = hh_days.filter("zero_day = 0")

periods = (usable.groupBy("household_id").agg(
    F.sum(((col("day") >= "2012-07-01") & (col("day") < "2013-01-01")).cast("int")).alias("h2_2012"),
    F.sum(((col("day") >= "2013-07-01") & (col("day") < "2014-01-01")).cast("int")).alias("h2_2013"),
    F.sum((F.year("day") == 2013).cast("int")).alias("usable_2013")))

hh_cov = (coverage.join(periods, "household_id", "left")
                  .fillna(0, subset=["h2_2012", "h2_2013", "usable_2013"])
                  .join(households_clean, "household_id"))

# 1. When did each group join the trial?
(hh_cov.groupBy(F.date_format("first_day", "yyyy-MM").alias("start_month"))
       .pivot("tariff").count().orderBy("start_month").show(40))

# 2. How many households qualify for the July-December comparison at different thresholds?
for t in [30, 60, 90, 120, 150]:
    (hh_cov.filter((col("h2_2012") >= t) & (col("h2_2013") >= t))
           .groupBy("tariff").count()
           .withColumn("min_days_each_half", F.lit(t))
           .show())

# 3. How many have enough 2013 data for the bill and carbon analysis?
for t in [90, 180, 300]:
    print(t, hh_cov.filter(col("usable_2013") >= t).groupBy("tariff").count().collect())

+-----------+----+----+
|start_month| Std| ToU|
+-----------+----+----+
|    2011-11|  51|  12|
|    2011-12| 303|  45|
|    2012-01| 129|  27|
|    2012-02| 216|  45|
|    2012-03| 328|  75|
|    2012-04| 624| 180|
|    2012-05|1257| 325|
|    2012-06| 461| 125|
|    2012-07| 422| 128|
|    2012-08|  13|   2|
|    2012-09| 196|  47|
|    2012-10| 409| 106|
|    2012-11|  14|NULL|
|    2012-12|   7|NULL|
|    2013-01|   1|NULL|
|    2013-02|   2|NULL|
|    2013-03|   1|NULL|
|    2013-04|   1|NULL|
|    2013-05|   1|NULL|
|    2013-07|   2|   1|
|    2013-09|NULL|   1|
|    2013-10|NULL|   3|
+-----------+----+----+

+------+-----+------------------+
|tariff|count|min_days_each_half|
+------+-----+------------------+
|   Std| 4195|                30|
|   ToU| 1074|                30|
+------+-----+------------------+

+------+-----+------------------+
|tariff|count|min_days_each_half|
+------+-----+------------------+
|   Std| 4146|                60|
|   ToU| 1062|                60|


Households are **flagged, not deleted**, so each analysis filters on the flag it needs and the day counts stay available. The total is 5,560 rather than 5,566: six households have no usable readings after cleaning (five never appear in the readings, and one time-of-use household lost all its days to the day-completeness rules in notebook 01).

In [32]:
#@title Save the households table with analysis flags

households_final = (hh_cov
    .withColumn("in_tariff_comparison", (col("h2_2012") >= 60) & (col("h2_2013") >= 60))
    .withColumn("in_2013_analysis", col("usable_2013") >= 180)
    .select("household_id", "tariff", "acorn", "acorn_group",
            "first_day", "last_day", "complete_days", "zero_days", "usable_days",
            "h2_2012", "h2_2013", "usable_2013",
            "in_tariff_comparison", "in_2013_analysis"))

# Check the counts by group
households_final.groupBy("tariff").agg(
    F.count("*").alias("households"),
    F.sum(col("in_tariff_comparison").cast("int")).alias("in_tariff_comparison"),
    F.sum(col("in_2013_analysis").cast("int")).alias("in_2013_analysis")).show()

# Save straight to Drive (small table, so writing to Drive directly is fine)
households_final.write.mode("overwrite").parquet("/content/drive/MyDrive/london-smart-meters/households")

+------+----------+--------------------+----------------+
|tariff|households|in_tariff_comparison|in_2013_analysis|
+------+----------+--------------------+----------------+
|   Std|      4438|                4146|            4230|
|   ToU|      1122|                1062|            1088|
+------+----------+--------------------+----------------+



## 9. The `household_day` table

One row per household per day, the table the analysis notebooks start from.

| Columns | Purpose |
|---|---|
| `kwh`, `kwh_peak`, `peak_share` | Daily consumption, and how much fell in the 16:00–19:00 evening peak |
| `zero_day`, `n_imputed`, `has_suspect_spike` | Quality flags carried from the readings |
| `year`, `month`, `day_of_week`, `is_weekend`, `is_holiday` | Calendar |
| `temp_mean`, `temp_min`, `temp_max` | Daily London temperature |
| `tariff`, `acorn_group`, `in_tariff_comparison`, `in_2013_analysis` | From the households table |

The checks: 3,490,380 rows, exactly the household-days kept in notebook 01, so no join duplicated or lost rows. `peak_share` is null only on the 14,954 zero days (no division by zero), and `holiday` is null on non-holidays; nothing else is null.

In [33]:
#@title Build the household_day table

# 1. Daily temperature from the hourly weather
daily_weather = (WeatherHourly_df
    .groupBy(F.to_date("time").alias("day"))
    .agg(F.round(F.avg("temperature"), 2).alias("temp_mean"),
         F.min("temperature").alias("temp_min"),
         F.max("temperature").alias("temp_max")))

# 2. Evening peak: readings from 16:00 up to (not including) 19:00
hour_of_day = F.hour("timestamp") + F.minute("timestamp") / 60
is_peak = (hour_of_day >= 16) & (hour_of_day < 19)

# 3. Add up each household's readings per day
household_day = (hh
    .groupBy("household_id", F.to_date("timestamp").alias("day"))
    .agg(F.round(F.sum("energy_kwh"), 3).alias("kwh"),
         F.round(F.sum(F.when(is_peak, col("energy_kwh")).otherwise(0)), 3).alias("kwh_peak"),
         F.sum(col("imputed").cast("int")).alias("n_imputed"),
         F.max(col("suspect_spike").cast("int")).cast("boolean").alias("has_suspect_spike"),
         F.max(col("zero_day").cast("int")).cast("boolean").alias("zero_day"))
    .withColumn("peak_share", F.when(col("kwh") > 0, F.round(col("kwh_peak") / col("kwh"), 4))))

# 4. Calendar columns
household_day = (household_day
    .withColumn("year", F.year("day"))
    .withColumn("month", F.month("day"))
    .withColumn("day_of_week", F.date_format("day", "E"))
    .withColumn("is_weekend", F.dayofweek("day").isin(1, 7)))

# 5. Join holidays, weather and household information
household_day = (household_day
    .join(holiday_df.withColumnRenamed("date", "day"), "day", "left")
    .withColumn("is_holiday", col("holiday").isNotNull())
    .join(daily_weather, "day", "left")
    .join(households_final.select("household_id", "tariff", "acorn_group",
                                  "in_tariff_comparison", "in_2013_analysis"),
          "household_id", "left"))

# 6. Check and save
print(f"Rows: {household_day.count():,}")    # should be 3,490,380
household_day.select([count(when(col(c).isNull(), c)).alias(c) for c in household_day.columns]).show(vertical=True)
household_day.show(5)

household_day.write.mode("overwrite").parquet("/content/drive/MyDrive/london-smart-meters/household_day")

Rows: 3,490,380
-RECORD 0-----------------------
 household_id         | 0       
 day                  | 0       
 kwh                  | 0       
 kwh_peak             | 0       
 n_imputed            | 0       
 has_suspect_spike    | 0       
 zero_day             | 0       
 peak_share           | 14954   
 year                 | 0       
 month                | 0       
 day_of_week          | 0       
 is_weekend           | 0       
 holiday              | 3412618 
 is_holiday           | 0       
 temp_mean            | 0       
 temp_min             | 0       
 temp_max             | 0       
 tariff               | 0       
 acorn_group          | 0       
 in_tariff_comparison | 0       
 in_2013_analysis     | 0       

+------------+----------+------+--------+---------+-----------------+--------+----------+----+-----+-----------+----------+-------+----------+---------+--------+--------+------+-----------+--------------------+----------------+
|household_id|       day|   k

## 10. Which half-hour does a reading cover?

Before matching readings to half-hourly prices and carbon intensity, it has to be settled whether a reading stamped 17:00 covers 17:00–17:30 (start of interval) or 16:30–17:00 (end). The trial's user guide says readings are in GMT but the convention isn't stated, so the data decides. Time-of-use households should use less than standard households in High-price half-hours and more in Low-price ones; the right alignment should show that contrast more sharply.

| Price band | ToU ÷ Std, start convention | ToU ÷ Std, end convention |
|---|---:|---:|
| High | 0.8723 | **0.8703** |
| Normal | 0.9130 | 0.9131 |
| Low | 0.9630 | **0.9634** |

The end-of-interval alignment gives the stronger contrast in both directions, and it agrees with notebook 01's finding that every household's final reading falls at 00:00. The evidence is consistent but modest (price events last at least three hours, so a one-slot shift only affects their edges), and the two versions differ by about 0.2%, so the choice barely moves any result. **Readings are treated as covering the half-hour ending at their timestamp.**

This is also a first result. Relative to Normal prices, time-of-use households used about 5% less in High-price half-hours and about 5% more in Low-price ones. They also used about 9% less than standard households even at Normal prices, which is more likely a difference between the groups (they volunteered) than an effect of the tariff. That's why notebook 03 compares the change from the 2012 baseline rather than the level.

In [34]:
#@title Which timestamp convention? Test both alignments against the tariff

households_2013 = households_final.filter("in_2013_analysis").select("household_id", "tariff")
readings_2013 = (hh.filter((F.year("timestamp") == 2013) & ~col("zero_day"))
                   .join(households_2013, "household_id"))

for convention, shift in [("start of interval", 0), ("end of interval", 30)]:
    # "end": a reading stamped 17:00 covers 16:30-17:00, so it gets the price slot that starts at 16:30
    prices = tariff_df.select((col("TariffDateTime") + F.expr(f"INTERVAL {shift} MINUTES")).alias("timestamp"),
                              col("Tariff").alias("price_band"))
    result = (readings_2013.join(prices, "timestamp")
              .groupBy("price_band").pivot("tariff").agg(F.avg("energy_kwh"))
              .withColumn("ToU_vs_Std", F.round(col("ToU") / col("Std"), 4)))
    print(f"\n===== Timestamps mark the {convention} =====")
    result.orderBy("price_band").show()


===== Timestamps mark the start of interval =====
+----------+-------------------+-------------------+----------+
|price_band|                Std|                ToU|ToU_vs_Std|
+----------+-------------------+-------------------+----------+
|      High|0.25907990783072393| 0.2259868801085259|    0.8723|
|       Low|0.21723508125997432|0.20920296006353434|     0.963|
|    Normal|0.21287716289809105|0.19435965649121548|     0.913|
+----------+-------------------+-------------------+----------+


===== Timestamps mark the end of interval =====
+----------+-------------------+-------------------+----------+
|price_band|                Std|                ToU|ToU_vs_Std|
+----------+-------------------+-------------------+----------+
|      High| 0.2575543570387799| 0.2241495383119616|    0.8703|
|       Low|0.21806129453429868|0.21008060387278968|    0.9634|
|    Normal|0.21286272437298623|0.19435809189983075|    0.9131|
+----------+-------------------+-------------------+----------+



## 11. Carbon and cost per household-day

Carbon intensity changes every half-hour, so emissions depend on *when* electricity was used: each reading's kWh is multiplied by its own half-hour's intensity and then summed per day. The same goes for the time-of-use bill. Under the end-of-interval convention, each reading is matched on `interval_start` = timestamp − 30 minutes, which is also how the NESO data and the tariff file label their half-hours. The evening peak is recalculated on the same basis.

- `kg_co2`: emissions using **average** grid intensity. The marginal estimate belongs to the load-shifting scenario later.
- `cost_tou_gbp`: the day's consumption priced on the 2013 time-of-use schedule (2013 days only). The one boundary reading per year with no tariff slot is priced at the Normal rate.
- `cost_flat_gbp`: the same consumption at the flat 14.228p/kWh, for every day.

Every reading matched a carbon intensity. 1,553,752 household-days have no time-of-use cost, which are the days outside 2013. The averages are plausible: about 10.2 kWh, 4.9 kg CO₂ (about 487 g/kWh, in line with the grid median) and £1.45 per household-day at the flat rate. The two cost averages aren't comparable as printed, since they cover different years and households; the proper bill comparison is notebook 03's first job.

In [35]:
#@title Add carbon and cost to household_day (end-of-interval convention)

# Carbon intensity and ToU prices, each labelled by the start of their half-hour
ci = genMix_df.select(col("DATETIME").alias("interval_start"),
                      col("CARBON_INTENSITY").alias("ci_g_per_kwh"))

price_of = F.create_map(F.lit("High"), F.lit(0.6720),
                        F.lit("Normal"), F.lit(0.1176),
                        F.lit("Low"), F.lit(0.0399))              # £/kWh
prices = tariff_df.select(col("TariffDateTime").alias("interval_start"),
                          price_of[col("Tariff")].alias("price_tou"))

FLAT = 0.14228                                                     # £/kWh, standard tariff

# Each reading covers the half-hour ending at its timestamp
r = (hh.withColumn("interval_start", col("timestamp") - F.expr("INTERVAL 30 MINUTES"))
       .join(F.broadcast(ci), "interval_start", "left")
       .join(F.broadcast(prices), "interval_start", "left"))

h = F.hour("interval_start") + F.minute("interval_start") / 60
in_2013 = F.year(F.to_date("timestamp")) == 2013

daily_add = (r.groupBy("household_id", F.to_date("timestamp").alias("day"))
    .agg(F.round(F.sum(F.when((h >= 16) & (h < 19), col("energy_kwh")).otherwise(0)), 3).alias("kwh_peak"),
         F.round(F.sum(col("energy_kwh") * col("ci_g_per_kwh")) / 1000, 4).alias("kg_co2"),
         F.sum(col("ci_g_per_kwh").isNull().cast("int")).alias("n_missing_ci"),
         F.round(F.sum(F.when(in_2013, col("energy_kwh") * F.coalesce(col("price_tou"), F.lit(0.1176)))), 4).alias("cost_tou_gbp"),
         F.round(F.sum(col("energy_kwh")) * FLAT, 4).alias("cost_flat_gbp")))

household_day = (household_day.drop("kwh_peak", "peak_share")
    .join(daily_add, ["household_id", "day"], "left")
    .withColumn("peak_share", F.when(col("kwh") > 0, F.round(col("kwh_peak") / col("kwh"), 4))))

# Checks
print(f"Rows: {household_day.count():,}")                       # still 3,490,380
household_day.agg(F.sum("n_missing_ci").alias("readings_without_ci"),
                  F.sum(col("cost_tou_gbp").isNull().cast("int")).alias("days_without_tou_cost")).show()
household_day.select("kwh", "kg_co2", "cost_flat_gbp", "cost_tou_gbp").summary("mean", "50%", "max").show()

household_day.write.mode("overwrite").parquet("/content/drive/MyDrive/london-smart-meters/household_day")

Rows: 3,490,380
+-------------------+---------------------+
|readings_without_ci|days_without_tou_cost|
+-------------------+---------------------+
|                  0|              1553752|
+-------------------+---------------------+

+-------+------------------+-----------------+------------------+-----------------+
|summary|               kwh|           kg_co2|     cost_flat_gbp|     cost_tou_gbp|
+-------+------------------+-----------------+------------------+-----------------+
|   mean|10.163285563463068|4.948086158727646|1.4460322284966387|1.413228449552517|
|    50%|             7.843|           3.7733|            1.1159|           0.9816|
|    max|           332.556|         188.2813|           47.3161|          80.9448|
+-------+------------------+-----------------+------------------+-----------------+



## 12. Save the supporting tables

The analysis notebooks need more than `household_day`: the half-hourly price schedule for the event-level tariff analysis, and the generation mix with its fuel columns for the marginal-emissions estimate. Everything goes to Drive, so no later notebook has to rerun an earlier one.

In [36]:
#@title Save the cleaned supporting tables to Drive

DRIVE = "/content/drive/MyDrive/london-smart-meters"

WeatherHourly_df.write.mode("overwrite").parquet(f"{DRIVE}/weather_hourly")       # with the 'interpolated' flag
holiday_df.write.mode("overwrite").parquet(f"{DRIVE}/bank_holidays")              # official list, 2011-2014
(tariff_df.withColumn("price_tou_gbp", price_of[col("Tariff")])                   # band and £/kWh
          .write.mode("overwrite").parquet(f"{DRIVE}/tariffs_2013"))
genMix_df.write.mode("overwrite").parquet(f"{DRIVE}/generation_mix")              # intensity and fuel columns

!ls /content/drive/MyDrive/london-smart-meters

bank_holidays	    household_day		       tariffs_2013
generation_mix	    households			       weather_hourly
halfhourly_cleaned  neso_generation_mix_2011_2014.csv


## Next

Notebook 03 starts from `household_day` and the households flags: the July–December comparison of time-of-use and standard households between 2012 and 2013, then the 2013 bills, separating what the price structure saved from what changed behaviour saved.